# Lipschitz of Sofplus (full autograd)

In [1]:
import torch
import torch.nn.functional as F
from torch.autograd.functional import jacobian as autograd_jacobian
import math


seed = 1
torch.manual_seed(seed)
ns = list(range(2, 16))      
steps = 10000                
lr = 1e-1                    
    
def Sofplus(x): 
    return torch.log1p(torch.exp(x))


for n in ns:
    g = torch.Generator().manual_seed(seed)
    
    x = (0.01 * torch.randn(n, generator=g, dtype=torch.float64)).requires_grad_(True)
    
    optimizer = torch.optim.Adam([x], lr=lr)

    for _ in range(steps):
            
        J = autograd_jacobian(Sofplus, x, create_graph=True)
                
        #K = torch.sqrt(torch.sum(J*J))   #<- it's ok but may have numeric error
        K = torch.linalg.norm(J, ord=2)   #<- more accurate              

        optimizer.zero_grad()
        (-K).backward()                                 
        optimizer.step()

    print(f"Sofplus n = {n:2d}  Estimated Lipschitz constant = {K.item():.6f} at x = {x.detach().numpy()}")

    

Sofplus n =  2  Estimated Lipschitz constant = 0.999999 at x = [1.37785062e+01 2.66924098e-03]
Sofplus n =  3  Estimated Lipschitz constant = 0.999999 at x = [1.37785062e+01 2.66924098e-03 6.16772582e-04]
Sofplus n =  4  Estimated Lipschitz constant = 0.999999 at x = [1.37785062e+01 2.66924098e-03 6.16772582e-04 6.21317329e-03]
Sofplus n =  5  Estimated Lipschitz constant = 0.999999 at x = [ 1.37785062e+01  2.66924098e-03  6.16772582e-04  6.21317329e-03
 -4.51905965e-03]
Sofplus n =  6  Estimated Lipschitz constant = 0.999999 at x = [ 1.37785062e+01  2.66924098e-03  6.16772582e-04  6.21317329e-03
 -4.51905965e-03 -1.66130235e-03]
Sofplus n =  7  Estimated Lipschitz constant = 0.999999 at x = [ 1.37785062e+01  2.66924098e-03  6.16772582e-04  6.21317329e-03
 -4.51905965e-03 -1.66130235e-03 -1.52276848e-02]
Sofplus n =  8  Estimated Lipschitz constant = 0.999999 at x = [ 1.37785062e+01  2.66924098e-03  6.16772582e-04  6.21317329e-03
 -4.51905965e-03 -1.66130235e-03 -1.52276848e-02  3.8168